# Beyond Words — Notebook 06: Hinglish evaluation of A and C, and the A–D comparison

**Research project:** Beyond Words — Cross-lingual Emotion and Sarcasm Analysis in Romanized Hinglish
**Notebook role:** (1) score the English-trained checkpoints A and C on the Hinglish test sets (zero-shot), (2) compare all four conditions on the shared six-class label space.

## Primary comparison

Six classes: **anger, disgust, fear, joy, sadness, surprise.** Gold-Neutral examples are dropped and the Neutral logit is masked for every condition, using the saved 7-way logits. Nothing is retrained. The original seven-class A/C scores are kept as supplementary English-baseline results (`macro_f1_7class`).

| Test set | Scored on | Notes |
|---|---|---|
| `goemo_test` | six classes | English, identical for all four conditions |
| `emosen_test` | six classes | Hinglish; in-domain for B/D, zero-shot for A/C. 13 Fear and 7 Surprise examples, so `macro_f1_core` is also reported (anger, disgust, joy, sadness where present: 4 classes on GoEmotions and EmoSen, 3 on HinEmo-9 because it has no Disgust) |
| `hinemo_test` | five classes | optional held-out Hinglish gold set (no Disgust); only if the prepared file exists |

## Contrasts (paired bootstrap over test examples, seed-averaged)

| Effect | Contrast |
|---|---|
| Hinglish adaptation, XLM-R | B − A |
| Hinglish adaptation, HingRoBERTa | D − C |
| Backbone, English-only | C − A |
| Backbone, adapted | D − B |
| Interaction | (D − C) − (B − A) |

The bootstrap interval reflects test-set sampling for the models actually trained. Run-to-run training variability is shown by the per-seed SDs and is not part of it. With three seeds per condition, treat everything here as descriptive.


## Section 1 — Installation

Package versions are pinned to the ones used for Condition A (`transformers==4.48.3`, `tokenizers==0.21.0`). **Restart the runtime after this cell**, then continue from Section 2. Compare the versions printed in Section 8b with Condition A's `experiment_metadata.json`.


In [1]:
import subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
         "transformers==4.48.3", "tokenizers==0.21.0", "sentencepiece",
         "accelerate>=1.2.0", "scikit-learn>=1.4.0", "seaborn>=0.13.0", "scipy>=1.12.0"],
        check=True,
    )
    print("Packages installed. RESTART THE RUNTIME NOW, then run from Section 2.")
else:
    print("Not in Colab: install transformers==4.48.3 tokenizers==0.21.0 sentencepiece "
          "accelerate scikit-learn seaborn scipy pandas numpy matplotlib in your environment.")


Packages installed. RESTART THE RUNTIME NOW, then run from Section 2.


## Section 2 — Imports

All imports are gathered in one cell so the notebook fails fast if a package is missing.  
`TOKENIZERS_PARALLELISM=false` suppresses a benign Hugging Face warning about forked processes.


In [3]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Standard library
import datetime
import json
import os
import platform
import random
import sys
import time
import warnings
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore", category=UserWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Scientific stack
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax as scipy_softmax

# scikit-learn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# Hugging Face Transformers
import transformers
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Platform     : {platform.platform()}")
print(f"In Colab     : {IN_COLAB}")


Python       : 3.13.16
PyTorch      : 2.14.1+cu130
Transformers : 4.48.3
NumPy        : 2.4.6
Pandas       : 2.2.3
Platform     : Linux-6.6.159+-x86_64-with-glibc2.39
In Colab     : True


## Section 3 — Paths, conditions and file locations

Finds each condition's saved outputs in the local project or in `MyDrive/Beyond-Words`. Conditions whose outputs are not found are skipped later with a message.

In [4]:
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/Beyond-Words")
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    IN_COLAB = False
DRIVE = Path("/content/drive/MyDrive/Beyond-Words")

def first_existing(*cands):
    for c in cands:
        if c is not None and Path(c).exists():
            return Path(c)
    return None

LABEL_ORDER = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
ID2LABEL    = {i: lbl for i, lbl in enumerate(LABEL_ORDER)}
LABEL2ID    = {lbl: i for i, lbl in enumerate(LABEL_ORDER)}
NUM_LABELS  = len(LABEL_ORDER)
SEEDS       = [42, 123, 2026]
CFG = {"use_fp16": True, "max_seq_len": 128, "eval_batch_size": 32}      # same as A/C for scoring

TAGS = {"A": "xlmr_english_ekman7_condA", "C": "hingroberta_english_ekman7_condC",
        "B": "xlmr_english_plus_emosen6_condB", "D": "hingroberta_english_plus_emosen6_condD"}
OUT_DIRNAME = {"A": "english_emotion", "C": "english_emotion_condC",
               "B": "hinglish_adapt_condB", "D": "hinglish_adapt_condD"}
MODEL_DIRNAME = {"A": "english_emotion", "C": "english_emotion_condC"}

ROOTS = {c: first_existing(PROJECT_ROOT / "outputs" / OUT_DIRNAME[c], DRIVE / "outputs" / OUT_DIRNAME[c]) for c in TAGS}
ZS_OUT = PROJECT_ROOT / "outputs" / "hinglish_zero_shot"
for sub in ("logits", "metrics"):
    (ZS_OUT / sub).mkdir(parents=True, exist_ok=True)
DATA_DIR    = PROJECT_ROOT / "data" / "processed"
EMOSEN_PATH = DATA_DIR / "emosen_dedup_frozen.csv"
HINEMO_PATH = DATA_DIR / "hinemo9_gold_hinglish_test.csv"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Output roots found:")
for c in "ABCD":
    print(f"  {c}: {ROOTS[c]}")

def logit_file(cond, set_name, seed):
    tag = TAGS[cond]
    if set_name == "goemo_test":
        name = f"test_logits_{tag}_seed{seed}.csv" if cond in "AC" else f"goemo_test_logits_{tag}_seed{seed}.csv"
        return (ROOTS[cond] / "logits" / name) if ROOTS[cond] else None
    if cond in "BD":
        return (ROOTS[cond] / "logits" / f"{set_name}_logits_{tag}_seed{seed}.csv") if ROOTS[cond] else None
    return ZS_OUT / "logits" / f"{set_name}_logits_{tag}_seed{seed}.csv"


Mounted at /content/drive
Output roots found:
  A: /content/drive/MyDrive/Beyond-Words/outputs/english_emotion
  B: /content/drive/MyDrive/Beyond-Words/outputs/hinglish_adapt_condB
  C: /content/drive/MyDrive/Beyond-Words/outputs/english_emotion_condC
  D: /content/drive/MyDrive/Beyond-Words/outputs/hinglish_adapt_condD


In [5]:
# ── Label-space constants (shared by training, evaluation and comparison) ─────
SIX_IDS  = (0, 1, 2, 3, 4, 5)       # anger, disgust, fear, joy, sadness, surprise (Neutral = 6 is never scored)
CORE_IDS = (0, 1, 3, 4)             # anger, disgust, joy, sadness: sensitivity set with enough support in EmoSen test
TEST_ALLOWED = {
    "goemo_test" : (0, 1, 2, 3, 4, 5),
    "emosen_test": (0, 1, 2, 3, 4, 5),
    "hinemo_test": (0, 2, 3, 4, 5),  # HinEmo-9 has no Disgust
}
# NOTE: "macro_f1_core" averages CORE_IDS that are present in the label space:
#       4 classes on goemo_test / emosen_test, 3 classes (anger, joy, sadness) on hinemo_test.


## Section 5 — GPU and environment inspection

### Why check the GPU first?

Google Colab's free T4 GPU has ~15 GB VRAM. Training XLM-R-base at batch size 16 uses approximately 7–9 GB. If CUDA is unavailable, training falls back to CPU, which is ~50× slower — one epoch may take several hours instead of ~12 minutes.

### Mixed precision (FP16)

Instead of storing all weights and activations as 32-bit floats, FP16 uses 16-bit floats for most operations. This approximately:
- Halves VRAM consumption
- Speeds up computation via NVIDIA Tensor Cores

PyTorch's `torch.cuda.amp.autocast()` context manager handles FP16 automatically and falls back to FP32 for numerically sensitive operations (e.g., softmax).

### Recovery if CUDA out-of-memory (OOM)

If you encounter a CUDA OOM error:
1. Reduce `CFG["train_batch_size"]` from 16 → 8
2. Increase `CFG["grad_accum_steps"]` from 1 → 2 (keeps effective batch = 16)
3. Reduce `CFG["max_seq_len"]` from 128 → 96
4. Restart the runtime and re-run from the top

Do not silently change these settings mid-run — document any deviation.


In [6]:
print("=" * 60)
print("ENVIRONMENT INSPECTION")
print("=" * 60)

CUDA_AVAILABLE = torch.cuda.is_available()

if CUDA_AVAILABLE:
    DEVICE     = torch.device("cuda")
    gpu_name   = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅  CUDA available")
    print(f"   GPU model  : {gpu_name}")
    print(f"   VRAM total : {gpu_mem_gb:.1f} GB")
    print(f"   Mixed prec : {'FP16 enabled' if CFG['use_fp16'] else 'disabled (CFG override)'}")
else:
    DEVICE = torch.device("cpu")
    CFG["use_fp16"] = False   # FP16 requires CUDA
    print("⚠️  CUDA not available — training on CPU.")
    print("   FP16 disabled automatically.")
    print("   WARNING: CPU training is ~50× slower than T4 GPU.")

print(f"\n   Selected device  : {DEVICE}")
print(f"   PyTorch version  : {torch.__version__}")
print(f"   Transformers     : {transformers.__version__}")
print(f"   Python           : {sys.version.split()[0]}")
print("=" * 60)


ENVIRONMENT INSPECTION
✅  CUDA available
   GPU model  : Tesla T4
   VRAM total : 15.6 GB
   Mixed prec : FP16 enabled

   Selected device  : cuda
   PyTorch version  : 2.14.1+cu130
   Transformers     : 4.48.3
   Python           : 3.13.16


## Section 4 — Evaluation code (same definitions as the training notebooks)

In [7]:
# ── EmotionDataset ────────────────────────────────────────────────────────
class EmotionDataset(Dataset):
    """PyTorch Dataset for seven-class emotion classification.

    Stores raw texts and example_ids alongside tensors for error analysis.
    Tokenisation uses truncation but NOT padding — dynamic padding is handled
    per-batch by EmotionDataCollator.
    """

    def __init__(
        self,
        texts       : List[str],
        labels      : List[int],
        example_ids : List[str],
        tokenizer   : AutoTokenizer,
        max_len     : int,
    ):
        self.texts       = texts
        self.labels      = labels
        self.example_ids = example_ids
        self.tokenizer   = tokenizer
        self.max_len     = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict:
        enc = self.tokenizer(
            self.texts[idx],
            truncation  = True,
            max_length  = self.max_len,
            padding     = False,     # padding handled by DataCollator
            return_tensors = "pt",
        )
        return {
            "input_ids"      : enc["input_ids"].squeeze(0),
            "attention_mask" : enc["attention_mask"].squeeze(0),
            "labels"         : torch.tensor(self.labels[idx], dtype=torch.long),
            "example_id"     : self.example_ids[idx],  # str — not a tensor
        }


# ── EmotionDataCollator ───────────────────────────────────────────────────
class EmotionDataCollator:
    """Collator that pads tensors per batch and passes string fields through."""

    def __init__(self, tokenizer: AutoTokenizer):
        self._pad = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

    def __call__(self, features: List[Dict]) -> Dict:
        # Pop string fields before passing to DataCollatorWithPadding
        example_ids = [f.pop("example_id") for f in features]
        batch       = self._pad(features)
        batch["example_id"] = example_ids
        return batch




In [8]:
# ── Metric computation ────────────────────────────────────────────────────
def compute_metrics(
    true_labels : List[int],
    pred_labels : List[int],
    prefix      : str = "",
) -> Dict[str, float]:
    """Compute accuracy, macro-F1, and weighted-F1.

    Parameters
    ----------
    true_labels : ground-truth integer class indices
    pred_labels : predicted integer class indices
    prefix      : key prefix, e.g. 'val_' or 'test_'

    Returns
    -------
    dict mapping metric name → float value
    """
    return {
        f"{prefix}accuracy"    : accuracy_score(true_labels, pred_labels),
        f"{prefix}macro_f1"    : f1_score(true_labels, pred_labels, average="macro",    zero_division=0),
        f"{prefix}weighted_f1" : f1_score(true_labels, pred_labels, average="weighted", zero_division=0),
    }


# ── Training epoch ────────────────────────────────────────────────────────
def train_one_epoch(
    model     : nn.Module,
    loader    : DataLoader,
    optimiser : torch.optim.Optimizer,
    scheduler,
    loss_fn   : nn.Module,
    cfg       : Dict,
    scaler    : Optional[torch.cuda.amp.GradScaler],
    epoch     : int,
) -> float:
    """One full training epoch. Returns mean training loss.

    Gradient accumulation note
    --------------------------
    Every mini-batch loss is divided by the *actual size of its accumulation
    group*, not by the global grad_accum_steps.  This matters for the final
    group of the epoch: if grad_accum_steps=4 but only 2 batches remain, those
    gradients are divided by 2 (the true group size), not 4.  Using the global
    value would under-scale the final update.

    group_start = (step // accum_steps) * accum_steps
    group_size  = min(accum_steps, n_batches - group_start)

    Gradients are flushed at every grad_accum_steps boundary *and* at the very
    last batch, ensuring nothing is discarded.  total_steps in the LR scheduler
    uses ceiling division (see run_training) to match this count exactly.
    """
    model.train()
    total_loss  = 0.0
    n_steps     = 0
    accum_steps = cfg["grad_accum_steps"]
    optimiser.zero_grad()

    n_batches = len(loader)
    for step, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        labels    = batch["labels"].to(DEVICE)

        # Actual number of batches in this accumulation group.
        # For a complete group this equals accum_steps.
        # For the final incomplete group it equals the remainder.
        group_start = (step // accum_steps) * accum_steps
        group_size  = min(accum_steps, n_batches - group_start)

        # Flush when we hit a group boundary or reach the very last batch.
        is_accum_step = (step + 1) % accum_steps == 0
        is_last_batch = (step + 1) == n_batches
        should_flush  = is_accum_step or is_last_batch

        if cfg["use_fp16"] and scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
                # Divide by group_size, not accum_steps, so the final
                # incomplete group receives correctly-scaled gradients.
                loss   = loss_fn(logits, labels) / group_size
            scaler.scale(loss).backward()
            if should_flush:
                scaler.unscale_(optimiser)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                scaler.step(optimiser)
                scaler.update()
                scheduler.step()
                optimiser.zero_grad()
        else:
            logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
            loss   = loss_fn(logits, labels) / group_size
            loss.backward()
            if should_flush:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                optimiser.step()
                scheduler.step()
                optimiser.zero_grad()

        # Scale the logged loss back to per-example scale for display.
        total_loss += loss.item() * group_size
        n_steps    += 1

        if step % 200 == 0:
            print(f"    Epoch {epoch} | Step {step:>5}/{n_batches} "
                  f"| Loss (scaled) {loss.item() * group_size:.4f} "
                  f"| group_size {group_size}")

    return total_loss / n_steps


# ── Evaluation epoch ──────────────────────────────────────────────────────
def evaluate(
    model   : nn.Module,
    loader  : DataLoader,
    loss_fn : nn.Module,
    cfg     : Dict,
) -> Tuple[float, List[int], List[int], np.ndarray, List[str]]:
    """Evaluate model on a DataLoader.

    Returns
    -------
    mean_loss   : float
    true_labels : List[int]
    pred_labels : List[int]
    all_logits  : np.ndarray of shape (N, 7)  — raw logits for all examples
    example_ids : List[str]
    """
    model.eval()
    total_loss  = 0.0
    n_steps     = 0
    true_labels = []
    pred_labels = []
    all_logits  = []
    example_ids = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels    = batch["labels"].to(DEVICE)
            ex_ids    = batch["example_id"]

            if cfg["use_fp16"]:
                with torch.cuda.amp.autocast():
                    logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
                    loss   = loss_fn(logits, labels)
            else:
                logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
                loss   = loss_fn(logits, labels)

            total_loss  += loss.item()
            n_steps     += 1
            true_labels .extend(labels.cpu().numpy().tolist())
            pred_labels .extend(logits.argmax(dim=-1).cpu().numpy().tolist())
            all_logits  .append(logits.float().cpu().numpy())
            example_ids .extend(ex_ids)

    return total_loss / n_steps, true_labels, pred_labels, np.vstack(all_logits), example_ids


print("✅  Training and evaluation functions defined.")


✅  Training and evaluation functions defined.


In [9]:
def compute_ece(
    true_labels : List[int],
    probs       : np.ndarray,
    n_bins      : int = 15,
) -> float:
    """Expected Calibration Error.

    Uses max-probability (confidence) as the binning dimension.

    Parameters
    ----------
    true_labels : ground-truth integer class indices (N,)
    probs       : softmax probability matrix (N, K)
    n_bins      : number of equal-width confidence bins (default 15)

    Returns
    -------
    float : ECE score (lower = better; 0 = perfect calibration)
    """
    max_probs   = probs.max(axis=1)
    pred_labels = probs.argmax(axis=1)
    correct     = (pred_labels == np.array(true_labels)).astype(float)
    bin_edges   = np.linspace(0.0, 1.0, n_bins + 1)
    ece         = 0.0
    n_total     = len(true_labels)

    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        in_bin = (max_probs >= lo) & (max_probs < hi)
        if in_bin.sum() == 0:
            continue
        acc_b = correct[in_bin].mean()
        con_b = max_probs[in_bin].mean()
        ece  += (in_bin.sum() / n_total) * abs(acc_b - con_b)

    return float(ece)


def compute_brier_score(
    true_labels : List[int],
    probs       : np.ndarray,
    num_classes : int,
) -> float:
    """Multiclass Brier Score.

    Parameters
    ----------
    true_labels : ground-truth integer class indices (N,)
    probs       : softmax probability matrix (N, K)
    num_classes : number of classes K

    Returns
    -------
    float : mean Brier score (lower = better; 0 = perfect)
    """
    n       = len(true_labels)
    one_hot = np.zeros((n, num_classes), dtype=float)
    for i, lbl in enumerate(true_labels):
        one_hot[i, lbl] = 1.0
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))




In [10]:
# ── Six-class (restricted) evaluation helpers ────────────────────────────────
# Label IDs are the frozen A/C ones: anger 0, disgust 1, fear 2, joy 3, sadness 4, surprise 5, neutral 6.
# The model keeps its 7-way head. For the primary comparison we (1) drop gold-Neutral examples and
# (2) mask every logit outside the allowed label space before argmax / softmax.
assert "TEST_ALLOWED" in globals() and "SIX_IDS" in globals(), "Run the label-space constants cell first."

def per_class_f1_fast(yt, yp, K=NUM_LABELS):
    cm = np.bincount(np.asarray(yt) * K + np.asarray(yp), minlength=K * K).reshape(K, K)
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    den = 2 * tp + fp + fn
    return np.divide(2 * tp, den, out=np.zeros(K), where=den > 0)

def restricted_metrics(true_ids, logits, allowed=SIX_IDS):
    """Metrics on the allowed label space. Gold labels outside it are dropped; logits outside it are masked."""
    t   = np.asarray(true_ids, dtype=int)
    lg  = np.asarray(logits, dtype=float)
    allowed = tuple(allowed)
    keep = np.isin(t, allowed)
    t, lg = t[keep], lg[keep]
    K = lg.shape[1]
    add = np.full(K, -1e9); add[list(allowed)] = 0.0
    pred        = (lg + add).argmax(axis=1)      # primary: Neutral / absent classes masked
    pred_strict = lg.argmax(axis=1)              # strict: a Neutral prediction counts as an error
    f1   = per_class_f1_fast(t, pred, K)
    f1_s = per_class_f1_fast(t, pred_strict, K)
    sup  = np.bincount(t, minlength=K)
    present = [c for c in allowed if sup[c] > 0]
    core    = [c for c in CORE_IDS if c in present]
    probs   = scipy_softmax(lg[:, list(allowed)], axis=1)
    y_local = np.searchsorted(np.array(allowed), t)
    out = {
        "n"                : int(len(t)),
        "n_classes"        : len(present),
        "accuracy"         : float((pred == t).mean()),
        "macro_f1"         : float(f1[present].mean()),
        "macro_f1_core"    : float(f1[core].mean()) if core else float("nan"),
        "n_core_classes"   : len(core),
        "macro_f1_strict"  : float(f1_s[present].mean()),
        "weighted_f1"      : float(np.average(f1[present], weights=sup[present])),
        "ece"              : compute_ece(y_local.tolist(), probs, n_bins=15),
        "brier"            : compute_brier_score(y_local.tolist(), probs, len(allowed)),
    }
    for i, lbl in enumerate(LABEL_ORDER):
        out[f"f1_{lbl}"]      = float(f1[i]) if i in present else float("nan")
        out[f"support_{lbl}"] = int(sup[i])
    return out

def read_logits_csv(path):
    d = pd.read_csv(path, dtype={"example_id": str})
    lg = d[[f"logit_{l}" for l in LABEL_ORDER]].values
    t  = d["true_label"].map(LABEL2ID).values
    return d["example_id"].tolist(), t, lg

def predictions_frame(ids, true_ids, logits):
    logits = np.asarray(logits, dtype=float)
    probs  = scipy_softmax(logits, axis=1)
    pred   = logits.argmax(axis=1)
    return pd.DataFrame({
        "example_id": list(ids),
        "true_label": [ID2LABEL[i] for i in true_ids],
        "pred_label": [ID2LABEL[i] for i in pred],
        "correct"   : [int(a) == int(b) for a, b in zip(true_ids, pred)],
        **{f"logit_{ID2LABEL[i]}": logits[:, i] for i in range(NUM_LABELS)},
        **{f"prob_{ID2LABEL[i]}" : probs[:, i]  for i in range(NUM_LABELS)},
    })


## Section 5 — Load the Hinglish test sets

The data-loader cell below holds the `EMOSEN_PERMISSION_CONFIRMED` gate (edit it there). The loader validates the CSV against the frozen dedup audit and drops `Others`.

In [17]:
!cp -r /content/drive/MyDrive/Beyond-Words/data/processed /content/Beyond-Words/data/

In [18]:
# ── EmoSen (frozen, deduplicated) and optional HinEmo-9 gold loaders ─────────
EMOSEN_PERMISSION_CONFIRMED = True  # set True ONLY with written confirmation covering this processing

SIX_NAMES = LABEL_ORDER[:6]
EXPECTED_EMOSEN_SIX = {   # retained rows per split after dedup, Others excluded (from the frozen audit)
    "train"     : {"anger": 2037, "disgust": 1016, "fear": 56, "joy": 3712, "sadness": 823, "surprise": 50},
    "validation": {"anger": 601,  "disgust": 55,   "fear": 4,  "joy": 961,  "sadness": 299, "surprise": 6},
    "test"      : {"anger": 677,  "disgust": 105,  "fear": 13, "joy": 1004, "sadness": 121, "surprise": 7},
}
EXPECTED_EMOSEN_OTHERS = {"train": 5828, "validation": 1036, "test": 1060}

def load_emosen_six(path):
    """Load the frozen EmoSen CSV, validate it against the frozen audit, drop Others, return six-class rows."""
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    pick = lambda opts: next((c for c in opts if c in raw.columns), None)
    c_split, c_id = pick(["split"]), pick(["id", "example_id"])
    c_text, c_lab = pick(["tweet_fixed", "text"]), pick(["emotion", "mapped_label", "label"])
    if None in (c_split, c_id, c_text, c_lab):
        raise ValueError(f"EmoSen CSV needs split/id/tweet_fixed/emotion columns; found {list(raw.columns)}")
    d = pd.DataFrame({
        "split"  : raw[c_split].str.strip().str.lower().replace({"val": "validation", "dev": "validation"}),
        "id"     : raw[c_id].str.strip(),
        "text"   : raw[c_text],
        "emotion": raw[c_lab].str.strip().str.lower(),
    })
    if set(d["split"]) != {"train", "validation", "test"}:
        raise ValueError(f"Unexpected EmoSen splits: {sorted(set(d['split']))}")
    if d.duplicated(["split", "id"]).any():
        raise ValueError("Duplicate (split, id) rows in the EmoSen CSV.")
    unknown = set(d["emotion"]) - set(SIX_NAMES) - {"others"}
    if unknown:
        raise ValueError(f"Unexpected EmoSen emotion labels: {sorted(unknown)}")
    if (d["text"].str.strip() == "").any():
        raise ValueError("Empty EmoSen text found.")

    others = d[d["emotion"] == "others"].groupby("split").size().to_dict()
    six    = d[d["emotion"] != "others"].copy()
    got    = {s: six[six["split"] == s].groupby("emotion").size().to_dict() for s in EXPECTED_EMOSEN_SIX}
    problems = []
    for s, exp in EXPECTED_EMOSEN_SIX.items():
        if got[s] != exp:
            problems.append(f"{s}: got {got[s]}, expected {exp}")
        if others.get(s, 0) != EXPECTED_EMOSEN_OTHERS[s]:
            problems.append(f"{s} Others: got {others.get(s, 0)}, expected {EXPECTED_EMOSEN_OTHERS[s]}")
    if problems:
        raise RuntimeError("EmoSen CSV does not match the frozen dedup audit:\n  " + "\n  ".join(problems))

    six["label_id"] = six["emotion"].map(LABEL2ID)
    return six.reset_index(drop=True), others

EXPECTED_HINEMO_RAW     = 393
EXPECTED_HINEMO_KEPT    = {"anger": 35, "fear": 49, "joy": 38, "sadness": 26, "surprise": 122}
EXPECTED_HINEMO_DROPPED = {"love": 41, "devotion": 31, "neutral": 26, "nostalgia": 25}

def load_hinemo_optional(path, strict=True):
    """Optional prepared HinEmo-9 gold Hinglish test file: columns id,text,label.
    Validates the raw file (393 rows, unique ids, non-empty text, expected label counts) before filtering to the
    five shared classes, so an accidentally replaced CSV cannot silently change the test set."""
    if not Path(path).exists():
        return None, {}
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    for c in ("id", "text", "label"):
        if c not in raw.columns:
            raise ValueError(f"HinEmo file needs columns id,text,label; found {list(raw.columns)}")
    lab = raw["label"].str.strip().str.lower()
    if strict:
        problems = []
        if len(raw) != EXPECTED_HINEMO_RAW:
            problems.append(f"rows: got {len(raw)}, expected {EXPECTED_HINEMO_RAW}")
        if raw["id"].str.strip().duplicated().any():
            problems.append("duplicate ids")
        if (raw["text"].str.strip() == "").any():
            problems.append("empty text")
        got = lab.value_counts().to_dict()
        want = {**EXPECTED_HINEMO_KEPT, **EXPECTED_HINEMO_DROPPED}
        if got != want:
            problems.append(f"label counts: got {got}, expected {want}")
        if problems:
            raise RuntimeError("HinEmo-9 CSV does not match the audited gold test split:\n  " + "\n  ".join(problems))
    dropped = lab[~lab.isin(SIX_NAMES)].value_counts().to_dict()
    d = raw[lab.isin(SIX_NAMES)].copy()
    d["label_id"] = lab[lab.isin(SIX_NAMES)].map(LABEL2ID)
    d = d.rename(columns={"id": "example_id"})[["example_id", "text", "label_id"]].reset_index(drop=True)
    return d, dropped


In [19]:
import hashlib
if not EMOSEN_PERMISSION_CONFIRMED:
    raise PermissionError("Set EMOSEN_PERMISSION_CONFIRMED = True (Section 3 data cell) only with written confirmation.")
if not EMOSEN_PATH.exists():
    raise FileNotFoundError("Upload emosen_dedup_frozen.csv to data/processed/ via the Colab Files panel (not Drive).")

hi_six, hi_others = load_emosen_six(EMOSEN_PATH)
_t = hi_six[hi_six["split"] == "test"]
EVAL_SETS = {"emosen_test": pd.DataFrame({"example_id": _t["id"].astype(str).values, "text": _t["text"].values,
                                          "label_id": _t["label_id"].values})}
_hn, _hn_dropped = load_hinemo_optional(HINEMO_PATH)
if _hn is not None:
    EVAL_SETS["hinemo_test"] = _hn
    print(f"HinEmo-9 gold test: {len(_hn)} rows (dropped outside shared classes: {_hn_dropped})")
else:
    print("hinemo_test file not found: continuing with emosen_test only.")
for n, d in EVAL_SETS.items():
    print(f"{n}: {len(d):,} rows;", d["label_id"].map(ID2LABEL).value_counts().reindex(LABEL_ORDER[:6]).fillna(0).astype(int).to_dict())


HinEmo-9 gold test: 270 rows (dropped outside shared classes: {'love': 41, 'devotion': 31, 'neutral': 26, 'nostalgia': 25})
emosen_test: 1,927 rows; {'anger': 677, 'disgust': 105, 'fear': 13, 'joy': 1004, 'sadness': 121, 'surprise': 7}
hinemo_test: 270 rows; {'anger': 35, 'disgust': 0, 'fear': 49, 'joy': 38, 'sadness': 26, 'surprise': 122}


## Section 6 — Zero-shot scoring of A and C on the Hinglish test sets

Loads each saved A and C checkpoint and writes its 7-way logits for `emosen_test` (and `hinemo_test` if present) in the same CSV format as the training notebooks. Resumable: an existing file is skipped. The English test logits already exist from the training runs and are not recomputed.


In [20]:
def predict_df(model_dir, df_set):
    tok   = AutoTokenizer.from_pretrained(str(model_dir))
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir)).to(DEVICE)
    ds = EmotionDataset(df_set["text"].tolist(), df_set["label_id"].astype(int).tolist(),
                        df_set["example_id"].astype(str).tolist(), tok, CFG["max_seq_len"])
    loader = DataLoader(ds, batch_size=CFG["eval_batch_size"], shuffle=False, collate_fn=EmotionDataCollator(tok),
                        num_workers=2, pin_memory=CUDA_AVAILABLE)
    _, t, _, lg, ids = evaluate(model, loader, nn.CrossEntropyLoss(), CFG)
    del model
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()
    return ids, t, lg

for cond in ("A", "C"):
    for seed in SEEDS:
        todo = {n: d for n, d in EVAL_SETS.items() if not logit_file(cond, n, seed).exists()}
        if not todo:
            print(f"{cond} seed {seed}: all Hinglish logits present, skipping.")
            continue
        mdir = first_existing(PROJECT_ROOT / "models" / MODEL_DIRNAME[cond] / TAGS[cond] / f"seed_{seed}",
                              DRIVE / "models" / MODEL_DIRNAME[cond] / TAGS[cond] / f"seed_{seed}")
        if mdir is None:
            print(f"WARNING: checkpoint for {cond} seed {seed} not found; skipped.")
            continue
        for n, d in todo.items():
            ids, t, lg = predict_df(mdir, d)
            predictions_frame(ids, t, lg).to_csv(logit_file(cond, n, seed), index=False)
            m = restricted_metrics(t, lg, TEST_ALLOWED[n])
            print(f"{cond} seed {seed} {n:<12} macro-F1 {m['macro_f1']:.4f}  core {m['macro_f1_core']:.4f}  acc {m['accuracy']:.4f}")


/tmp/ipykernel_1465/3696150764.py:146: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


A seed 42 emosen_test  macro-F1 0.2873  core 0.4256  acc 0.6715


/tmp/ipykernel_1465/3696150764.py:146: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


A seed 42 hinemo_test  macro-F1 0.3722  core 0.3826  acc 0.4370


/tmp/ipykernel_1465/3696150764.py:146: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


A seed 123 emosen_test  macro-F1 0.2757  core 0.4105  acc 0.6741


... (14 outputs left out)

/tmp/ipykernel_1465/3696150764.py:146: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


C seed 2026 emosen_test  macro-F1 0.3619  core 0.5349  acc 0.7732


/tmp/ipykernel_1465/3696150764.py:146: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


C seed 2026 hinemo_test  macro-F1 0.4794  core 0.4692  acc 0.5370


## Section 7 — Four-condition comparison

For every test set where all seeds of a condition exist, this prints mean ± SD over seeds, then the paired-bootstrap contrasts. Conditions with missing files are listed and skipped, so the cell can be run before B and D finish.


In [21]:
CONDS = ["A", "B", "C", "D"]
NAMES = {"A": "A  XLM-R, English", "B": "B  XLM-R, Eng+Hinglish", "C": "C  HingRoBERTa, English", "D": "D  HingRoBERTa, Eng+Hinglish"}
SETS_TO_COMPARE = ["goemo_test", "emosen_test", "hinemo_test"]
B_BOOT = 2000

def gather(set_name):
    data = {}
    for c in CONDS:
        paths = {s: logit_file(c, set_name, s) for s in SEEDS}
        miss = [s for s, p in paths.items() if p is None or not Path(p).exists()]
        if miss:
            print(f"  [{set_name}] condition {c}: missing seeds {miss}; skipped")
            continue
        data[c] = {s: read_logits_csv(p) for s, p in paths.items()}
    return data

all_rows, contrast_rows = [], []
for set_name in SETS_TO_COMPARE:
    print("\n" + "=" * 78 + f"\nTEST SET: {set_name}\n" + "=" * 78)
    data = gather(set_name)
    if len(data) < 2:
        print("  fewer than two conditions available; nothing to compare.")
        continue
    allowed = TEST_ALLOWED[set_name]
    ref_ids = next(iter(next(iter(data.values())).values()))[0]
    ref_t   = next(iter(next(iter(data.values())).values()))[1]
    for c, per_seed in data.items():
        for s, (ids, t, lg) in per_seed.items():
            assert ids == ref_ids and np.array_equal(t, ref_t), f"{set_name}: {c} seed {s} differs from the reference test order"
    keep = np.isin(ref_t, allowed)
    t_k  = ref_t[keep]
    K    = NUM_LABELS
    add  = np.full(K, -1e9); add[list(allowed)] = 0.0
    sup  = np.bincount(t_k, minlength=K)
    present = [c_ for c_ in allowed if sup[c_] > 0]
    core    = [c_ for c_ in CORE_IDS if c_ in present]
    preds = {c: {s: (lg[keep] + add).argmax(1) for s, (ids, t, lg) in per_seed.items()} for c, per_seed in data.items()}

    # per-seed metrics and mean ± SD
    for c, per_seed in data.items():
        for s, (ids, t, lg) in per_seed.items():
            m = restricted_metrics(t, lg, allowed); m.update(cond=c, seed=s, test_set=set_name)
            if set_name == "goemo_test":
                m["macro_f1_7class"] = float(f1_score(t, lg.argmax(1), average="macro", zero_division=0))
            all_rows.append(m)
    sub = pd.DataFrame([r for r in all_rows if r["test_set"] == set_name])
    cols = ["macro_f1", "macro_f1_core", "macro_f1_strict", "accuracy", "weighted_f1", "ece", "brier"] + \
           (["macro_f1_7class"] if set_name == "goemo_test" else [])
    print(f"n scored = {len(t_k)}  classes = {[ID2LABEL[i] for i in present]}")
    print(f"{'condition':<30}" + "".join(f"{c_:>20}" for c_ in cols))
    for c in data:
        g_ = sub[sub.cond == c]
        print(f"{NAMES[c]:<30}" + "".join(f"{g_[c_].mean():>12.4f}±{g_[c_].std():.4f}" for c_ in cols))

    # paired bootstrap
    rng = np.random.default_rng(20261115)
    n = len(t_k)
    F = {c: np.zeros((B_BOOT, K)) for c in data}
    for b in range(B_BOOT):
        idx = rng.integers(0, n, n); tb = t_k[idx]
        for c in data:
            F[c][b] = np.mean([per_class_f1_fast(tb, preds[c][s][idx], K) for s in SEEDS], axis=0)
    P = {c: np.mean([per_class_f1_fast(t_k, preds[c][s], K) for s in SEEDS], axis=0) for c in data}

    def _contrast(label, plus, minus, kind="diff"):
        for metric, cls in (("macro_f1", present), ("macro_f1_core", core)):
            if not cls:
                continue
            if kind == "diff":
                if plus not in data or minus not in data:
                    return
                pt = P[plus][cls].mean() - P[minus][cls].mean()
                bs = F[plus][:, cls].mean(1) - F[minus][:, cls].mean(1)
            else:
                if not all(x in data for x in "ABCD"):
                    return
                pt = (P["D"][cls].mean() - P["C"][cls].mean()) - (P["B"][cls].mean() - P["A"][cls].mean())
                bs = (F["D"][:, cls].mean(1) - F["C"][:, cls].mean(1)) - (F["B"][:, cls].mean(1) - F["A"][:, cls].mean(1))
            lo, hi = np.percentile(bs, [2.5, 97.5])
            contrast_rows.append({"test_set": set_name, "contrast": label, "metric": metric,
                                  "delta": pt, "ci_low": lo, "ci_high": hi})
    _contrast("Adaptation, XLM-R (B - A)", "B", "A")
    _contrast("Adaptation, HingRoBERTa (D - C)", "D", "C")
    _contrast("Backbone, English-only (C - A)", "C", "A")
    _contrast("Backbone, adapted (D - B)", "D", "B")
    _contrast("Interaction ((D-C) - (B-A))", None, None, kind="interaction")

    cr = pd.DataFrame([r for r in contrast_rows if r["test_set"] == set_name])
    if len(cr):
        print(f"\nPaired bootstrap contrasts ({B_BOOT} resamples; test-sampling uncertainty only):")
        with pd.option_context("display.float_format", "{:+.4f}".format, "display.width", 160):
            print(cr.drop(columns="test_set").to_string(index=False))
    pcd = pd.DataFrame({"label": [ID2LABEL[i] for i in present], "support": [int(sup[i]) for i in present],
                        **{f"f1_{c}": [P[c][i] for i in present] for c in data}})
    print("\nPer-class F1 (mean over seeds):")
    with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 160):
        print(pcd.to_string(index=False))
    pcd.to_csv(ZS_OUT / "metrics" / f"per_class_f1_{set_name}.csv", index=False)

if all_rows:
    pd.DataFrame(all_rows).to_csv(ZS_OUT / "metrics" / "comparison_ABCD_by_seed.csv", index=False)
if contrast_rows:
    pd.DataFrame(contrast_rows).to_csv(ZS_OUT / "metrics" / "comparison_ABCD_contrasts.csv", index=False)
print("\nSaved comparison tables to", ZS_OUT / "metrics")



TEST SET: goemo_test
n scored = 3357  classes = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
condition                                 macro_f1       macro_f1_core     macro_f1_strict            accuracy         weighted_f1                 ece               brier     macro_f1_7class
A  XLM-R, English                   0.7217±0.0107      0.7161±0.0107      0.7146±0.0118      0.8153±0.0061      0.8180±0.0053      0.0642±0.0078      0.2829±0.0062      0.6273±0.0068
B  XLM-R, Eng+Hinglish              0.7259±0.0011      0.7218±0.0029      0.7166±0.0026      0.8156±0.0003      0.8187±0.0003      0.0648±0.0019      0.2850±0.0041      0.6306±0.0053
C  HingRoBERTa, English             0.7328±0.0017      0.7294±0.0054      0.7254±0.0020      0.8252±0.0009      0.8272±0.0006      0.0556±0.0095      0.2659±0.0051      0.6406±0.0046
D  HingRoBERTa, Eng+Hinglish        0.7260±0.0046      0.7266±0.0038      0.7164±0.0064      0.8198±0.0020      0.8221±0.0018      0.0584±0.0088      0.

## Section 7b — Why a macro-F1 difference can hide large changes

Macro-F1 can look unchanged while individual classes move a lot in opposite directions. For each test set this prints, per condition (all seeds pooled): the gold label share, the share of **predictions** per class, per-class **recall**, and the row-normalised confusion matrix. Use it before reading any "no effect" result. A class whose recall collapses while another class is over-predicted indicates a label-prior or label-definition mismatch between training and test data, not a lack of effect. It uses saved logits only.


In [22]:
diag_rows = []
for set_name in SETS_TO_COMPARE:
    data = gather(set_name)
    if not data:
        continue
    allowed = list(TEST_ALLOWED[set_name])
    ref_t = next(iter(next(iter(data.values())).values()))[1]
    keep = np.isin(ref_t, allowed); t_k = ref_t[keep]
    K = NUM_LABELS
    add = np.full(K, -1e9); add[allowed] = 0.0
    gold_share = np.bincount(t_k, minlength=K) / len(t_k)
    recs = {}
    for c, per_seed in data.items():
        pred_all = np.concatenate([(lg[keep] + add).argmax(1) for (_, _, lg) in per_seed.values()])
        t_rep = np.tile(t_k, len(per_seed))
        cm = np.bincount(t_rep * K + pred_all, minlength=K * K).reshape(K, K)
        rs = cm.sum(1)
        recs[c] = {"pred_share": np.bincount(pred_all, minlength=K) / len(pred_all),
                   "recall": np.divide(np.diag(cm), rs, out=np.full(K, np.nan), where=rs > 0), "cm": cm}
    print("\n" + "=" * 90 + f"\n{set_name}: gold share, predicted share and recall per class\n" + "=" * 90)
    hdr = f"{'class':<10}{'gold':>7}" + "".join(f"{'pred ' + c:>9}{'rec ' + c:>8}" for c in recs)
    print(hdr)
    for i in allowed:
        row = f"{ID2LABEL[i]:<10}{gold_share[i]:>7.3f}" + "".join(
            f"{recs[c]['pred_share'][i]:>9.3f}{recs[c]['recall'][i]:>8.3f}" for c in recs)
        print(row)
        diag_rows.append({"test_set": set_name, "class": ID2LABEL[i], "gold_share": gold_share[i],
                          **{f"pred_share_{c}": recs[c]["pred_share"][i] for c in recs},
                          **{f"recall_{c}": recs[c]["recall"][i] for c in recs}})
    if set_name != "goemo_test":
        for c in recs:
            cm = recs[c]["cm"][np.ix_(allowed, allowed)].astype(float)
            cmn = np.divide(cm, cm.sum(1, keepdims=True), out=np.zeros_like(cm), where=cm.sum(1, keepdims=True) > 0)
            print(f"\nCondition {c}: row-normalised confusion on {set_name} (rows = gold, columns = predicted)")
            print(pd.DataFrame(cmn, index=[ID2LABEL[i] for i in allowed], columns=[ID2LABEL[i] for i in allowed]).round(2).to_string())
if diag_rows:
    pd.DataFrame(diag_rows).to_csv(ZS_OUT / "metrics" / "prediction_distribution_by_class.csv", index=False)



goemo_test: gold share, predicted share and recall per class
class        gold   pred A   rec A   pred B   rec B   pred C   rec C   pred D   rec D
anger       0.170    0.173   0.717    0.172   0.714    0.172   0.730    0.166   0.714
disgust     0.023    0.026   0.605    0.025   0.627    0.024   0.610    0.023   0.588
fear        0.024    0.028   0.783    0.027   0.767    0.028   0.787    0.029   0.787
joy         0.553    0.527   0.884    0.521   0.879    0.532   0.896    0.527   0.887
sadness     0.084    0.088   0.696    0.090   0.698    0.088   0.696    0.086   0.694
surprise    0.145    0.158   0.776    0.165   0.798    0.156   0.783    0.169   0.801

emosen_test: gold share, predicted share and recall per class
class        gold   pred A   rec A   pred B   rec B   pred C   rec C   pred D   rec D
anger       0.351    0.337   0.620    0.054   0.084    0.309   0.677    0.109   0.237
disgust     0.054    0.001   0.000    0.353   0.848    0.005   0.013    0.254   0.787
fear        0.0

## Section 8 — Save

In [23]:
import shutil
if IN_COLAB:
    from google.colab import files
    dst = DRIVE / "outputs" / "hinglish_zero_shot"
    shutil.copytree(ZS_OUT, dst, dirs_exist_ok=True); print("Copied:", ZS_OUT, "->", dst)
    shutil.make_archive("/content/Beyond-Words-hinglish_zero_shot", "zip", ZS_OUT)
    files.download("/content/Beyond-Words-hinglish_zero_shot.zip")
else:
    print("Outputs are on local disk under", ZS_OUT)


Copied: /content/Beyond-Words/outputs/hinglish_zero_shot -> /content/drive/MyDrive/Beyond-Words/outputs/hinglish_zero_shot


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>